# Lekcja 2.7 — eksperymenty: heurystyki dla zbierania paczek

Uruchamiaj komórki po kolei. Przed każdym eksperymentem zapisz swoje przewidywanie w komórce „Twoje przewidywanie” (kliknij ją dwukrotnie, żeby edytować), a dopiero potem uruchom kod.

Pierwsza komórka wybiera implementację:

- `"mine"` używa Twojego `task.py`; najpierw rozwiąż część B zadania;
- `"reference"` używa implementacji wzorcowej kursu. Wpisz `IMPLEMENTATION = "reference"`, jeśli chcesz uruchomić notebook przed skończeniem zadania.

W trybie `"reference"` „Twoja” heurystyka to wersja wzorcowa kursu, czyli *h*<sub>3</sub>.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    from task import parcel_heuristic
else:
    from introai.world.reference import parcel_heuristic

Funkcja `heuristics(warehouse)` zwraca słownik heurystyk do porównania: *h* = 0, kandydatów *h*<sub>1</sub>–*h*<sub>4</sub> z lekcji i Twoją. Funkcja `solve` uruchamia A\* z jedną z nich i zwraca plan, liczbę rozwinięć oraz czas w milisekundach. Problem zbierania paczek pochodzi z wersji wzorcowej kursu (lekcja 1.3).

In [ ]:
import time
from pathlib import Path

import matplotlib.pyplot as plt

from introai.search.reference import a_star
from introai.testing.search import true_costs
from introai.trace import TraceLimitExceeded, record
from introai.world.reference import CANDIDATE_HEURISTICS, ParcelPickupProblem
from introai.world.warehouse import WarehouseMap

MAPS = Path.cwd().parents[2] / "data" / "maps"
aisles = WarehouseMap.from_file(MAPS / "aisles.txt")
CELLS = [(3, 1), (5, 4), (9, 21), (13, 15), (1, 20), (7, 13)]


def yours(state):
    try:
        return parcel_heuristic(state)
    except NotImplementedError:
        raise RuntimeError(
            'parcel_heuristic is not written yet - set IMPLEMENTATION = "reference" in the '
            "first cell to run the experiments first"
        ) from None


def heuristics(warehouse):
    named = {"h = 0": lambda state: 0}
    for name in ("h1", "h2", "h3", "h4"):
        named[name] = CANDIDATE_HEURISTICS[name](warehouse)
    named["yours"] = yours
    return named


def solve(problem, h, limit=500_000):
    """A* with the heuristic h: (goal node, expansions, milliseconds)."""
    start = time.perf_counter()
    try:
        with record(limit=limit, kinds={"expand"}) as trace:
            node = a_star(problem, h)
    except TraceLimitExceeded:
        raise RuntimeError(f"A* expanded more than {limit} nodes") from None
    return node, len(trace.of_kind("expand")), (time.perf_counter() - start) * 1000

## Eksperyment 1: od 1 do 6 paczek

A\* zbiera pierwsze 1, 2, …, 6 paczek z listy `CELLS` na mapie z alejkami, za każdym razem z inną heurystyką.

**Twoje przewidywanie.** Ułóż heurystyki od tej, która rozwija najwięcej węzłów, do tej, która rozwija najmniej. Która z nich da za długi plan?

*Moje przewidywanie:* …

In [ ]:
counts = list(range(1, len(CELLS) + 1))
work = {}
print(f"{'parcels':>7} " + " ".join(f"{name:>12}" for name in heuristics(aisles)))
for k in counts:
    warehouse = aisles.with_parcels(CELLS[:k])
    cells = []
    best = None
    for name, h in heuristics(warehouse).items():
        node, expanded, _ = solve(ParcelPickupProblem(warehouse), h)
        work.setdefault(name, []).append(expanded)
        best = node.path_cost if best is None else min(best, node.path_cost)
        cells.append((expanded, node.path_cost))
    row = [f"{e}/{c:g}" + ("!" if c > best else "") for e, c in cells]
    print(f"{k:>7} " + " ".join(f"{text:>12}" for text in row))
print("expansions/plan length; ! marks a plan longer than the shortest one")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
series = [
    ("h = 0", "#2a78d6", "o"),
    ("h1", "#eb6834", "s"),
    ("h2", "#1baf7a", "^"),
    ("h3", "#eda100", "D"),
    ("yours", "#e87ba4", "v"),
]
for name, color, marker in series:
    ax.plot(
        counts,
        work[name],
        color=color,
        linewidth=2,
        marker=marker,
        markersize=8,
        markeredgecolor="#fcfcfb",
        markeredgewidth=1.5,
        label=name,
    )
# Direct labels at the line ends, nudged apart on the log scale so they never overlap.
label_y = None
for name, *_ in sorted(series, key=lambda item: work[item[0]][-1]):
    label_y = work[name][-1] if label_y is None else max(work[name][-1], label_y * 1.3)
    ax.annotate(
        name,
        xy=(counts[-1], work[name][-1]),
        xytext=(counts[-1] + 0.15, label_y),
        textcoords="data",
        va="center",
        color="#0b0b0b",
    )
ax.set_yscale("log")
ax.set_title("Nodes expanded by A* (admissible heuristics only)", loc="left", color="#0b0b0b")
ax.set_xlabel("number of parcels", color="#52514e")
ax.set_ylabel("nodes expanded (log scale)", color="#52514e")
ax.margins(x=0.12)
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
ax.legend(frameon=False, loc="upper left", labelcolor="#0b0b0b")
fig.tight_layout()

**Do przemyślenia.** Dlaczego *h*<sub>4</sub> rozwija najmniej węzłów, a mimo to jest bezużyteczna, jeśli zależy nam na najkrótszym planie? O ile *h*<sub>3</sub> jest lepsza od *h*<sub>2</sub> przy sześciu paczkach? Gdzie na wykresie jest Twoja heurystyka?

## Eksperyment 2: dominacja

Porównujemy heurystyki stan po stanie: we wszystkich 1240 stanach osiągalnych przy trzech paczkach.

**Twoje przewidywanie.** Czy *h*<sub>3</sub> jest większa lub równa *h*<sub>2</sub> w każdym stanie? Jak wypada Twoja heurystyka w porównaniu z *h*<sub>3</sub>?

*Moje przewidywanie:* …

In [ ]:
warehouse = aisles.with_parcels(CELLS[:3])
states = list(true_costs(ParcelPickupProblem(warehouse)))
named = heuristics(warehouse)


def compare(first, second):
    larger = sum(named[first](s) > named[second](s) for s in states)
    smaller = sum(named[first](s) < named[second](s) for s in states)
    print(
        f"{first} vs {second}: larger in {larger}, smaller in {smaller}, "
        f"equal in {len(states) - larger - smaller} of {len(states)} states"
    )


compare("h3", "h2")
compare("h2", "h1")
compare("yours", "h3")

**Do przemyślenia.** Co oznacza wynik „smaller in 0”? Dlaczego *h*<sub>3</sub> nigdy nie jest mniejsza od *h*<sub>2</sub>? Jeśli Twoja heurystyka jest w części stanów większa, a w części mniejsza od *h*<sub>3</sub>, to jaką heurystykę warto zbudować z obu?

## Eksperyment 3: rozwinięcia a czas

Przy sześciu paczkach mierzymy, ile milisekund zajmuje A\* z każdą heurystyką (mediana z trzech uruchomień).

**Twoje przewidywanie.** Czy kolejność według czasu będzie taka sama jak kolejność według liczby rozwinięć?

*Moje przewidywanie:* …

In [ ]:
warehouse = aisles.with_parcels(CELLS)
for name, h in heuristics(warehouse).items():
    runs = sorted((solve(ParcelPickupProblem(warehouse), h) for _ in range(3)), key=lambda r: r[2])
    node, expanded, ms = runs[1]
    print(f"{name:>6}: {expanded:>6} expansions, {ms:>7.1f} ms, plan {node.path_cost:g} actions")

**Do przemyślenia.** Czy któraś heurystyka rozwija mniej węzłów, a mimo to nie jest szybsza? Ile kosztuje policzenie *h*<sub>3</sub> dla jednego stanu w porównaniu z *h*<sub>1</sub>? Czasy zależą od komputera; porównuj je między sobą, a nie z liczbami kolegów.

## Podsumowanie

Wróć do pytań kontrolnych w lekcji. Po własnej próbie możesz porównać swoje wyniki i wnioski z odpowiedziami w `solutions/02_search/07_designing_heuristics/answers.pl.md`.